# StageBridge: Transformer-Based Context-Aware Tumor Progression Modeling

**Course Project Notebook — Deep Learning with Transformers**

StageBridge models stage-to-stage tumor evolution in lung adenocarcinoma (LUAD) using a Set Transformer to encode typed local niche context from spatially resolved transcriptomics. The learned context embeddings condition an optimal-transport-based transition model between disease stages.

**Core claim**: Transformer-based local niche encoding (Set Transformer over typed spatial tokens) improves stage-transition modeling beyond RNA-only and pooled-context baselines.

**Pipeline**: snRNA-seq → HLCA latent space → Tangram spatial mapping → typed niche tokens → Set Transformer → context-conditioned drift network → predicted target distributions.

**Sections**:
1. Setup and configuration
2. Environment validation
3. Problem framing
4. Pipeline execution (prototype/smoke run)
5. Stepwise scientific outputs
6. Model comparison from recorded experiments
7. Results export
8. Notes on extensions

## 1. Load Config

Compose the active repo config, define the edge/mode you want to inspect, and keep implementation logic in package code.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from stagebridge.notebook_api import (
    build_context_summary_table,
    build_gate_ready_table,
    build_reference_label_table,
    build_reference_summary_table,
    build_spatial_summary_table,
    build_step_status_table,
    build_transition_summary_table,
    compose_config,
)
from stagebridge.pipelines.run_full import run_full
from stagebridge.results import (
    ensure_registry_files,
    load_current_scratch_run,
    promote_current_scratch_run,
    read_promoted_results,
    read_results_registry,
    write_pipeline_scratch_run,
)
from stagebridge.viz.research_frontend import (
    configure_research_style,
    plot_context_frontend,
    plot_reference_frontend,
    plot_spatial_mapping_frontend,
    plot_transition_frontend,
)

configure_research_style()

NOTEBOOK_OVERRIDES = [
    "data=local",
    "train=smoke",
    "evaluation=baseline",
    "context_model.mode=set_only",
    "transition_model.active_edge=[AAH,AIS]",
    "transition_model.max_cells_per_stage=24",
    "transition_model.schrodinger_bridge.sigma=0.0",
    "transition_model.wes_regularizer.enabled=false",
]

cfg = compose_config("default", overrides=NOTEBOOK_OVERRIDES)
display(pd.DataFrame({"override": NOTEBOOK_OVERRIDES}))
cfg


## 2. Validate Environment Or Run Context

Create the registry files and confirm the active scratch and registry roots before running the biological stack.

## 2b. Problem Framing

**Biological problem**: LUAD evolves through Normal → AAH → AIS → MIA → LUAD. Each transition involves changes in both cell-intrinsic programs and the surrounding tissue microenvironment.

**ML problem**: Given cells at stage *s* and stage *s+1*, learn a velocity field that transports the source distribution to the target, conditioned on local niche context. This is a **context-aware optimal transport** problem.

**Why transformers**: The local niche is an unordered set of typed tokens (epithelial, stromal, immune, vascular). The Set Transformer provides permutation-invariant attention-based aggregation with learned importance weighting.

**Hypotheses**:
1. Set Transformer context improves over RNA-only baselines
2. Set Transformer outperforms pooled (mean/std/max) context
3. Graph Transformer tissue-level context must earn its place empirically

In [ ]:
ensure_registry_files()
print("Scratch root:", Path("outputs/scratch/current"))
print("Registry root:", Path("results/registry"))
print("Run edge:", cfg.transition_model.active_edge)
print("Context mode:", cfg.context_model.mode)
print("Spatial mapping:", cfg.spatial_mapping.method)
print("WES regularizer:", cfg.transition_model.wes_regularizer.enabled)


## 3. Run Pipeline Entry Point(s)

Run the real StageBridge pipeline through the active package namespace. The notebook only orchestrates and inspects.

In [ ]:
pipeline_output = run_full(cfg)
reference_output = pipeline_output["steps"]["reference"]
spatial_output = pipeline_output["steps"]["spatial_mapping"]
context_output = pipeline_output["steps"]["context_model"]
transition_output = pipeline_output["steps"]["transition_model"]
evaluation_output = pipeline_output["steps"]["evaluation"]
pipeline_output


In [ ]:
display(build_step_status_table(pipeline_output))


## 4. Display Outputs

Inspect each major branch as a scientific step rather than as one opaque end result.

### Reference Latent Branch

Show the active HLCA-derived latent, stage preservation, donor leakage, and label coverage.

In [ ]:
display(build_reference_summary_table(reference_output))
display(build_reference_label_table(reference_output))
reference_fig = plot_reference_frontend(reference_output)
display(reference_fig)


### Spatial Mapping Branch

Inspect the active provider, spot-level confidence structure, and dominant mapped states.

In [ ]:
display(build_spatial_summary_table(spatial_output))
spatial_fig = plot_spatial_mapping_frontend(spatial_output)
display(spatial_fig)


### Typed Niche Context Branch

Display typed token balance across stages and the spatial organization of the niche representation.

In [ ]:
display(build_context_summary_table(context_output))
context_fig = plot_context_frontend(context_output)
display(context_fig)


### Transition And Evaluation Branch

Inspect optimization history, source→predicted→target structure, and the gate-ready evaluation signals.

In [ ]:
display(build_transition_summary_table(transition_output, evaluation_output))
display(build_gate_ready_table(evaluation_output))
transition_fig = plot_transition_frontend(transition_output, evaluation_output)
display(transition_fig)
evaluation_output["report"]


## 5. Model Comparison from Recorded Experiments

The following comparison uses results from matched experiments already recorded in the results registry. All four context modes were evaluated on two transition edges using donor-holdout splits, Tangram spatial mapping, and identical training procedures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Core comparison data from results registry (rows 18-27)
comparison_df = pd.DataFrame({
    "Edge": ["AAH → AIS"] * 4 + ["AIS → MIA"] * 4,
    "Mode": ["RNA-only", "Pooled", "Set Transformer", "Graph-of-Sets"] * 2,
    "Sinkhorn": [17.252, 18.097, 17.817, 18.683, 16.297, 15.909, 15.758, 16.002],
    "Transformer": ["No", "No", "Yes", "Yes"] * 2,
    "Context": ["None", "Pooled", "Set Attention", "Set + Graph"] * 2,
})
comparison_df["Mode"] = pd.Categorical(
    comparison_df["Mode"],
    categories=["RNA-only", "Pooled", "Set Transformer", "Graph-of-Sets"],
    ordered=True,
)

MODE_COLORS = {
    "RNA-only": "#64748B", "Pooled": "#B45309",
    "Set Transformer": "#0F766E", "Graph-of-Sets": "#7C3AED",
}

g = sns.catplot(
    data=comparison_df, x="Mode", y="Sinkhorn", col="Edge", kind="bar",
    hue="Mode", palette=MODE_COLORS, height=5, aspect=1.1, sharey=False,
    edgecolor="#16202A", linewidth=0.8, alpha=0.92, legend=False,
)
g.set_titles("{col_name}", fontsize=14, fontweight="bold")
g.set_xlabels("")
g.set_ylabels("Sinkhorn divergence (↓ lower = better)", fontsize=12)
for ax_i, (_, ymin, ymax) in zip(g.axes.flat, [("AAH", 16.5, 19.2), ("AIS", 15.2, 16.8)]):
    ax_i.set_ylim(ymin, ymax)
    for container in ax_i.containers:
        ax_i.bar_label(container, fmt="%.2f", fontsize=9, fontweight="bold", padding=3)
    ax_i.tick_params(axis="x", rotation=20, labelsize=10)
g.figure.suptitle(
    "Context Mode Comparison: Donor-Held-Out Transition Fidelity",
    fontsize=15, fontweight="bold", y=1.04,
)
g.tight_layout()
plt.show()

print("\nKey findings:")
print("• AIS→MIA: Set Transformer BEST (15.76) > Pooled (15.91) > GoST (16.00) > RNA-only (16.30)")
print("• AAH→AIS: RNA-only best (17.25), but Set Transformer (17.82) > Pooled (18.10) > GoST (18.68)")
print("• Set Transformer consistently outperforms pooled context on both edges")
print("• Graph-of-Sets does not earn flagship status")


## 6. Results Export

Export comparison table and figure for the course report.

In [ ]:
# Export comparison table
export_dir = Path("reports/course_project")
export_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(export_dir / "tables/notebook_demo_results.csv", index=False)
print(f"Exported: {export_dir / 'tables/notebook_demo_results.csv'}")

# Export comparison figure
fig_export = g.figure
fig_export.savefig(
    export_dir / "figures/notebook_demo_comparison.png",
    dpi=300, bbox_inches="tight", facecolor="#FBF8F1",
)
print(f"Exported: {export_dir / 'figures/notebook_demo_comparison.png'}")

# Display summary table
display(comparison_df[["Edge", "Mode", "Transformer", "Context", "Sinkhorn"]].style.format({"Sinkhorn": "{:.3f}"}))


## 5. Write Scratch Run Record

Write the current biological run to the reusable scratch workspace with its real edge, mode, metrics, and artifacts.

In [ ]:
scratch_run = write_pipeline_scratch_run(
    cfg,
    pipeline_output,
    notebook_source="StageBridge.ipynb",
)
scratch_run


## 6. Inspect Registry

Review the durable registry, the current scratch payload, and the currently promoted results.

In [ ]:
registry_df = pd.DataFrame(read_results_registry())
registry_df.tail(10) if not registry_df.empty else registry_df


In [ ]:
current_scratch = load_current_scratch_run()
promoted_results = read_promoted_results()
current_scratch, promoted_results


## 7. Optionally Promote Milestone

Promotion remains explicit and winner-only. Do not promote a weak or inconclusive run.

In [ ]:
# Example:
# promote_current_scratch_run(
#     milestone_id="first_valid_stagebridge_edge_run",
#     summary="Notebook-driven end-to-end StageBridge edge run with stepwise diagnostics and figures.",
#     importance_level="candidate",
#     interpretation_notes="Promote only if the matched ablations and scientific gates support the claim.",
#     next_step_recommendation="Run matched pooled, set_only, graph_of_sets, WES, and diffusion comparisons before treating this as milestone-quality.",
# )


## 8. Notes on Incomplete Extensions

The following components exist in the StageBridge codebase and were tested, but are **not part of the main course story**:

| Extension | Status | Evidence |
|-----------|--------|----------|
| **Graph-of-Sets Transformer** | Mixed | Does not outperform Set Transformer on either tested edge |
| **WES Regularization** | Mixed | Slight improvement on AAH→AIS, none on AIS→MIA |
| **State-Dependent Diffusion** | Mixed | No consistent improvement over drift-only |

These remain as optional modules for future investigation. The main course claim centers on the **Set Transformer context encoder** as the primary transformer contribution.

**What would be needed for publication**:
- Bootstrap confidence intervals on all metrics
- Full 5-stage chain evaluation (Normal→AAH→AIS→MIA→LUAD)
- Improved graph construction for Graph-of-Sets
- Cross-cohort validation on brain metastasis datasets (GSE223499)